In [2]:
import tiktoken
import numpy as np
# from senten

In [3]:
def softmax(x, axis= 0):
    # subtract max for numerical stability

    # across rows
    x_shifted = x - np.max(x, axis = axis, keepdims=True)
    
    exp_x = np.exp(x_shifted)
    
    return exp_x / np.sum(exp_x, axis=axis, keepdims=True)

In [4]:
# axis= 0 # column
# axis= 1 # row

In [5]:
def self_attention(query , key, value):
    # q, k, v are np arrays and are of same order
    # writing codea asuming they are horizontal matrices
    dk = key.shape[1]
    a = (query @ key.T) / np.sqrt(dk) # dot product but for a number of words will be orchestrsted by cross product

    a = softmax(a , axis = 1)
    
    b = a @ value
    
    return b
    

In [6]:
class MultiHeadAttention:

    def __init__(self, embed_dim, num_heads, masked = False):
        self.embed_dim = embed_dim 
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.masked = masked

        self.WQ = [np.random.randn(embed_dim, self.head_dim) for _ in range(num_heads)]
        self.WK = [np.random.randn(embed_dim, self.head_dim) for _ in range(num_heads)]
        self.WV = [np.random.randn(embed_dim, self.head_dim) for _ in range(num_heads)]

        self.WO = np.random.randn(embed_dim, embed_dim)

    def self_attention(self,query , key, value):
        score = (query @ key.T) / np.sqrt(self.head_dim)
        # print("Before Mask : " ,score)
        if (self.masked):
            print("Masked is turned true")
            r,c = score.shape
            # print ("r= ", r , " c = ", c)
            for i in range(r):
                for j in range(i+1, c):
                    score[i][j] = -np.inf
            # print("After Mask : ", score)



        attention = softmax(score , axis = 1)
        # print("After Softmax: ", attention)
        
        b = attention @ value
        return b
            
    
    def forward(self, query, key, value):
        heads =  []
        for i in range(self.num_heads):

            q = query @ self.WQ[i]
            k = key @ self.WK[i]
            v = value @ self.WV[i]
            head = self.self_attention(q, k, v)
            heads.append(head)

        context_awared = np.concatenate(heads, axis = 1)

        output = context_awared @ self.WO # without this step all the heads will be separate. But this combines the information of all the heads
        return output

positional encoder

In [7]:
# based on formula

def positional_encoder_f(pos, embedding_dimension):

    if (embedding_dimension % 2 != 0 ):
        raise ValueError("Embedding dimension must be even.")
    encoded_pos = []    
    for i in range (int((embedding_dimension / 2))) :
        a = np.sin(pos/ np.power(10000, ((2 * i) / embedding_dimension)))
        b = np.cos(pos/ np.power(10000, ((2 * i) / embedding_dimension)))
        encoded_pos += [a,b]

    return encoded_pos

In [8]:
def get_positional_encoding_for_fixed_number_of_words(number_of_words, embedding_dimension):
    encoded_positions = []
    for i in range (number_of_words):
        encoded_positions += [positional_encoder_f(i,embedding_dimension)]
        # print(encoded_positions, "\n")

    # print("\n\n", encoded_positions)
    return np.vstack([encoded_positions])
    
    

layer normalization


In [9]:
class LayerNorm:
    def __init__ (self, embed_dim):

        self.gamma = np.ones(embed_dim) # initializing to 1 because initially no scaling 
        self.beta = np.zeros(embed_dim) # initializing to 0 because initially no shifting

    def forward (self, x):
        m = np.mean(x, axis = 1, keepdims=True)
        s = np.std(x, axis = 1, keepdims=True)

        normalized = (x - m ) / (s + 1e-5) # incase preventing divison by zero

        return self.gamma * normalized + self.beta

feed forward network


In [10]:
import torch
from torch import nn
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


In [11]:
class FeedForward(nn.Module):
    def __init__(self ,embedding_dimension, hidden_dimension):
        super(FeedForward, self).__init__()
        self.linear1 = nn.Linear(embedding_dimension, hidden_dimension)
        self.linear2 = nn.Linear(hidden_dimension, embedding_dimension)

    def forward(self, x):
        x = self.linear1(x)
        x = torch.relu(x)
        x = self.linear2(x)
        return x

In [12]:
class EncoderBlock:

    def __init__(self, embed_dim, num_heads, d_ff):
        self.embed_dim = embed_dim
        self.num_heads = num_heads

        self.attention = MultiHeadAttention(self.embed_dim, self.num_heads)

        self.norm = LayerNorm(embed_dim)
        self.norm2 = LayerNorm(embed_dim)
        self.ffn = FeedForward(embed_dim, d_ff )

    

    def forward(self, x):

        # x is with positional encoding information

        attention_output = self.attention.forward(x,x,x) # self attention

        x = x + attention_output

        x = self.norm.forward(x) # learnable beta and gamma

        x_tensor = torch.tensor(x, dtype=torch.float32)

        ff_o = self.ffn.forward(x_tensor)

        x = x + (ff_o).detach().numpy()

        x = self.norm2.forward(x) # learnable beta gamma 2

        return x


In [82]:
import tiktoken
enc = tiktoken.encoding_for_model("gpt-4") # it uses BPE
print(enc.n_vocab)
tokens = (enc.encode("Hello Mangesh here i am  <|endoftext|>", allowed_special={"<|endoftext|>"}))


100277


In [53]:
np.random.randn(2,4)[0]

array([ 0.17452407,  1.81250579,  1.06296821, -0.96979156])

In [60]:
class Embeddings:

    def __init__(self, vocab_size = 100277, embed_dim = 512):

        self.vocab_size = vocab_size
        self.embed_dim = embed_dim
        self.embedding_matrix = np.random.randn(vocab_size, embed_dim)
        print("Shaoe of embeddig martrix: " ,self.embedding_matrix.shape)
        


    # vocab_size = enc.n_vocab # it has inbuilt special token <|endoftext|>

    def get_embedidngs_with_pe(self, tokens):
        embeddings = []
        token_count = len(tokens)
        for t in tokens:
            embeddings += [self.embedding_matrix[t]]
        embeddings = np.concatenate([embeddings], axis = 1)
        pe = get_positional_encoding_for_fixed_number_of_words(token_count, self.embed_dim)

        return embeddings + pe
    


In [61]:
embeddings_pe = embedding.get_embedidngs_with_pe(tokens)

In [62]:
embeddings_pe.shape

(8, 512)

### Decoder Block

In [108]:
class DecoderBlock:

    def __init__(self, embed_dim, num_heads, d_ff):
        
        self.masked_attention = MultiHeadAttention(embed_dim, num_heads, masked = True)

        self.norm1 = LayerNorm(embed_dim)

        self.cross_attention = MultiHeadAttention(embed_dim, num_heads, masked = False)
        self.norm2  = LayerNorm(embed_dim)

        self.ffn = FeedForward(embed_dim, d_ff)
        self.norm3 = LayerNorm(embed_dim)
    

    def forward (self, decoder_input, encoder_output):


        masked_output = self.masked_attention.forward(decoder_input, decoder_input, decoder_input)
        # print("masked op:", masked_output.shape)
        x = decoder_input + masked_output

        x = self.norm1.forward(x)

        cross_att_op = self.cross_attention.forward(x, encoder_output, encoder_output)

        x = cross_att_op + x

        x = self.norm2.forward(x)

        x_tensor = torch.tensor(x, dtype=torch.float32)

        ff_o = self.ffn.forward(x_tensor)

        x = x + (ff_o).detach().numpy()

        x = self.norm3.forward(x)
        # print("FInal decoder block op:" ,x.shape)
        return x
        


In [109]:
class Encoder:

    def __init__(self, num_layers, embed_dim, num_heads, d_ff):

        self.layers  = [EncoderBlock(embed_dim, num_heads, d_ff) for _ in range(num_layers)]

    def forward(self, x):

        for layer in self.layers:
            x = layer.forward(x)

        return x
    
class Decoder:
    def __init__(self,num_layers ,embed_dim, num_heads, d_ff):

        self.layers = [DecoderBlock(embed_dim, num_heads, d_ff) for _ in range(num_layers)]

    def forward (self, x, encoder_output):
        for layer in self.layers:
            x = layer.forward(x, encoder_output)

        return x
    


In [110]:
encoder = Encoder(2, 12, 3, 12)
decoder = Decoder(2, 12, 3, 12)

In [111]:
tokens

[9906, 60148, 4385, 1618, 602, 1097, 256, 100257]

In [112]:
embeddingg = Embeddings(vocab_size= 100258, embed_dim= 12)

Shaoe of embeddig martrix:  (100258, 12)


In [113]:
embeddingg.embedding_matrix.shape

(100258, 12)

In [114]:
embedding_pe = embeddingg.get_embedidngs_with_pe(tokens)
embedding_pe.shape

(8, 12)

In [115]:
encoder_op = encoder.forward(embedding_pe)

In [116]:
decoder = Decoder(6, 12, 3 , 12)


In [117]:
decoder_emb = Embeddings(20000, 12)

Shaoe of embeddig martrix:  (20000, 12)


In [118]:
deco_ip = enc.encode("Hello")

In [119]:
dec_emb = decoder_emb.get_embedidngs_with_pe(deco_ip)

In [120]:
decode_op = decoder.forward(dec_emb, encoder_op)

Masked is turned true
Masked is turned true
Masked is turned true
Masked is turned true
Masked is turned true
Masked is turned true
Masked is turned true
Masked is turned true
Masked is turned true
Masked is turned true
Masked is turned true
Masked is turned true
Masked is turned true
Masked is turned true
Masked is turned true
Masked is turned true
Masked is turned true
Masked is turned true


In [121]:
decode_op.shape

(1, 12)

In [122]:
class OutputProj(nn.Module):
    def __init__(self, embed_dim, vocab_size):
        super().__init__()

        self.linear = nn.Linear(embed_dim, vocab_size)

    def forward(self, x):
        if not isinstance(x, torch.Tensor):
            x = torch.tensor(x, dtype=torch.float32)
            
        x = self.linear(x)
        return x
    

linearProjection = OutputProj(embed_dim=12, vocab_size= 100227)         
x = linearProjection.forward(decode_op)
x_probs = torch.softmax(x , dim = 1)

In [ ]:
predicted_ids = torch.argmax(x_probs, dim = 1)

tensor([78453])

In [128]:
text= enc.decode(predicted_ids.tolist())
text

' 연'